In [ ]:
# 셀 1. GPU 확인
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되어 있지 않습니다. Colab 런타임 유형을 GPU(T4 권장)로 변경하세요.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)

print("Device:", device)
print("GPU:", gpu_name)

print("GPU 확인 완료. 유료 GPU이면 30 epoch 실험에 더 적합합니다.")


PyTorch version: 2.11.0+cu128
CUDA available: True
Device: cuda
GPU: NVIDIA L4
GPU 확인 완료. 유료 GPU이면 30 epoch 실험에 더 적합합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 셀 2. 패키지 설치
!pip install -q transformers accelerate sentencepiece pillow tqdm gdown pandas

In [ ]:
# 셀 3. 기본 설정
import os
import re
import json
import math
import random
import tarfile
import zipfile
import shutil
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from tqdm.auto import tqdm
import pandas as pd

from transformers import (
    Pix2StructProcessor,
    Pix2StructForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "google/pix2struct-base"
PROMPT = "이 식품 패키지 이미지에서 원재료명을 추출하세요."

MAX_SAMPLES = 3000
TRAIN_RATIO = 0.8

EPOCHS = 30
BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
LEARNING_RATE = 3e-5
GRAD_CLIP_NORM = 1.0

EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4

MAX_PATCHES = 512
MAX_TARGET_LENGTH = 256
MAX_GENERATION_LENGTH = 256
EVAL_NUM_BEAMS_LIST = [1, 3]

# 이미지 전처리 옵션
USE_IMAGE_PREPROCESSING = True
USE_TRAIN_AUGMENTATION = False
RESIZE_LONG_SIDE = 1024
CONTRAST_FACTOR = 1.20
SHARPNESS_FACTOR = 1.10
DENOISE_SIZE = 3

ALLERGY_KEYWORDS = [
    "밀", "우유", "대두", "땅콩", "메밀", "고등어", "게", "새우",
    "돼지고기", "복숭아", "토마토", "호두", "알류", "계란", "난황", "난백",
    "아황산", "이산화황", "조개류", "굴", "전복", "홍합", "오징어"
]


TAR_ID = "1yt9MAel2hGzf9EwxXASzSpLjFnPIxJ7X"
LABEL_ID = "1WPKwrRuY5SHoj2ImpUYVshPbXdMpdBC8"

WORK_DIR = Path("/content/pix2struct_food_v2")
DOWNLOAD_DIR = WORK_DIR / "download"
DATA_DIR = WORK_DIR / "data"
IMAGE_DIR = DATA_DIR / "solo"
OUTPUT_DIR = Path("/content/drive/MyDrive/pix2struct_food_v2_result")

TAR_PATH = DOWNLOAD_DIR / "solo.tar"
LABEL_PATH = DOWNLOAD_DIR / "label_log.txt"

BEST_MODEL_DIR = OUTPUT_DIR / "pix2struct_v2_best"
FINAL_MODEL_DIR = OUTPUT_DIR / "pix2struct_v2_final"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MODEL_NAME:", MODEL_NAME)
print("PROMPT:", PROMPT)
print("MAX_SAMPLES:", MAX_SAMPLES)
print("OUTPUT_DIR:", OUTPUT_DIR)


MODEL_NAME: google/pix2struct-base
PROMPT: 이 식품 패키지 이미지에서 원재료명을 추출하세요.
MAX_SAMPLES: 3000
OUTPUT_DIR: /content/drive/MyDrive/pix2struct_food_v2_result


In [ ]:
# 셀 4. solo.tar, label_log.txt 직접 다운로드
import gdown

if not TAR_PATH.exists():
    print("solo.tar 다운로드 시작")
    gdown.download(id=TAR_ID, output=str(TAR_PATH), quiet=False)
else:
    print("solo.tar already exists:", TAR_PATH)

if not LABEL_PATH.exists():
    print("label_log.txt 다운로드 시작")
    gdown.download(id=LABEL_ID, output=str(LABEL_PATH), quiet=False)
else:
    print("label_log.txt already exists:", LABEL_PATH)

if not TAR_PATH.exists():
    raise FileNotFoundError("solo.tar 다운로드 실패")
if not LABEL_PATH.exists():
    raise FileNotFoundError("label_log.txt 다운로드 실패")

print("TAR_PATH:", TAR_PATH, TAR_PATH.stat().st_size)
print("LABEL_PATH:", LABEL_PATH, LABEL_PATH.stat().st_size)


solo.tar already exists: /content/pix2struct_food_v2/download/solo.tar
label_log.txt already exists: /content/pix2struct_food_v2/download/label_log.txt
TAR_PATH: /content/pix2struct_food_v2/download/solo.tar 7862419968
LABEL_PATH: /content/pix2struct_food_v2/download/label_log.txt 1234452


In [ ]:
# 셀 5. solo.tar 압축 해제 및 이미지 확인
if IMAGE_DIR.exists():
    shutil.rmtree(IMAGE_DIR)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("압축 해제 시작:", TAR_PATH)

if str(TAR_PATH).endswith(".tar"):
    with tarfile.open(TAR_PATH, "r") as tar:
        tar.extractall(IMAGE_DIR)
elif str(TAR_PATH).endswith(".zip"):
    with zipfile.ZipFile(TAR_PATH, "r") as zip_ref:
        zip_ref.extractall(IMAGE_DIR)
else:
    raise ValueError("지원하지 않는 압축 형식입니다. tar 또는 zip만 지원합니다.")

image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
image_paths = [path for path in IMAGE_DIR.rglob("*") if path.suffix.lower() in image_extensions]
image_paths = sorted(image_paths, key=lambda p: p.name)

print("Image count:", len(image_paths))
if len(image_paths) == 0:
    raise RuntimeError("solo.tar 안에서 이미지 파일을 찾지 못했습니다.")

print("이미지 예시:")
for path in image_paths[:10]:
    print(path.name, "|", path)


압축 해제 시작: /content/pix2struct_food_v2/download/solo.tar


/tmp/ipykernel_19704/2997734049.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(IMAGE_DIR)


Image count: 3000
이미지 예시:
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.430/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.222/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.2370/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.389/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.1519/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.716/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.1629/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.1055/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.1503/step0.camera.png
step0.camera.png | /content/pix2struct_food_v2/data/solo/sequence.1926/step0.camera.png


In [ ]:
# 셀 6. label_log.txt 라벨 읽기
def clean_ingredient_text(text):
    text = text.strip()
    text = re.sub(r"^Ingredients\s*:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Ingredient\s*:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^원재료명\s*:\s*", "", text)
    text = re.sub(r"^성분\s*:\s*", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def parse_label_line(line):
    line = line.strip()
    if not line:
        return None

    image_pattern = r"([^\s]+\.(?:png|jpg|jpeg|bmp|webp))"
    image_match = re.search(image_pattern, line, flags=re.IGNORECASE)
    if image_match:
        filename = Path(image_match.group(1)).name
        remaining_text = line[image_match.end():].strip()
        return {"filename": filename, "text": clean_ingredient_text(remaining_text), "raw": line}

    return {"filename": None, "text": clean_ingredient_text(line), "raw": line}

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    raw_label_lines = f.readlines()

parsed_labels = []
for line in raw_label_lines:
    parsed = parse_label_line(line)
    if parsed is not None and parsed["text"]:
        parsed_labels.append(parsed)

filename_label_count = sum(1 for item in parsed_labels if item["filename"] is not None)
no_filename_label_count = sum(1 for item in parsed_labels if item["filename"] is None)

print("Label count:", len(parsed_labels))
print("Labels with filename:", filename_label_count)
print("Labels without filename:", no_filename_label_count)
if len(parsed_labels) == 0:
    raise RuntimeError("유효한 라벨을 찾지 못했습니다.")

print("라벨 예시:")
for item in parsed_labels[:5]:
    print(item)


Label count: 3000
Labels with filename: 0
Labels without filename: 3000
라벨 예시:
{'filename': None, 'text': '복숭아베이스, 천일염, 향료, 캐슈넛, 정제수, 복합조미식품, 마조람분말, 복합조미식품, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘', 'raw': 'Ingredients: 복숭아베이스, 천일염, 향료, 캐슈넛, 정제수, 복합조미식품, 마조람분말, 복합조미식품, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘'}
{'filename': None, 'text': "쇼트닝, 연꽃유함유가공품, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 유당(우유), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 복숭아추출물, 카다몬에멀전, 5'-이노신산디나트륨, 효모식품, 삼씨앗, 마늘분말, 찹쌀, 산도조절제, 토코페롤(혼합형), 이산화황, 은행나무견과, 소두구분말, 고등어", 'raw': "Ingredients: 쇼트닝, 연꽃유함유가공품, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 유당(우유), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 복숭아추출물, 카다몬에멀전, 5'-이노신산디나트륨, 효모식품, 삼씨앗, 마늘분말, 찹쌀, 산도조절제, 토코페롤(혼합형), 이산화황, 은행나무견과, 소두구분말, 고등어"}
{'filename': None, 'text': '정제소금, 복숭아, 사과산, 바닐라향루, 로커스트콩검, 정제소금, 분말결정포도당, 게추출물, 닭고기, 혼합제제, 밀가루, 백설탕, -폴리리

In [ ]:
# 셀 7. 이미지-라벨 매칭
image_by_name = {path.name: path for path in image_paths}
samples = []

if filename_label_count > 0:
    print("파일명 기준으로 이미지와 라벨을 매칭합니다.")
    missing_images = []
    for item in parsed_labels:
        filename = item["filename"]
        if filename is None:
            continue
        if filename in image_by_name:
            samples.append({"image_path": image_by_name[filename], "target_text": item["text"], "filename": filename})
        else:
            missing_images.append(filename)
    print("Matched sample count:", len(samples))
    print("Missing image count:", len(missing_images))
else:
    print("라벨에 파일명이 없으므로 이미지 정렬 순서와 라벨 줄 순서로 매칭합니다.")
    pair_count = min(len(image_paths), len(parsed_labels))
    if len(image_paths) != len(parsed_labels):
        print("[경고] 이미지 수와 라벨 수가 다릅니다.")
        print("Image count:", len(image_paths))
        print("Label count:", len(parsed_labels))
        print("매칭 가능한 개수만 사용:", pair_count)
    for image_path, item in zip(image_paths[:pair_count], parsed_labels[:pair_count]):
        samples.append({"image_path": image_path, "target_text": item["text"], "filename": image_path.name})

print("Final matched sample count:", len(samples))
if len(samples) == 0:
    raise RuntimeError("최종 매칭 샘플이 0개입니다.")

print("매칭 예시:")
for sample in samples[:5]:
    print(sample["filename"], "=>", sample["target_text"][:120])


라벨에 파일명이 없으므로 이미지 정렬 순서와 라벨 줄 순서로 매칭합니다.
Final matched sample count: 3000
매칭 예시:
step0.camera.png => 복숭아베이스, 천일염, 향료, 캐슈넛, 정제수, 복합조미식품, 마조람분말, 복합조미식품, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿
step0.camera.png => 쇼트닝, 연꽃유함유가공품, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 유당(우유), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 복숭아추출물, 카다몬에멀전, 
step0.camera.png => 정제소금, 복숭아, 사과산, 바닐라향루, 로커스트콩검, 정제소금, 분말결정포도당, 게추출물, 닭고기, 혼합제제, 밀가루, 백설탕, -폴리리신, 스테비올배당체, 토코페롤(혼합형), 치즈, 고등어, 물엿, 유청분말, 흑
step0.camera.png => 정제소금, 기타과당, 볶음양파분말, 대두함유, 물엿, 효모식품, 아몬드, 기타가공품, 육두구분말, 곡류가공품, 해바라기유(외국산), 향료, 유청, 토마토, 정제수, 우유, 천일염, 쇼트닝, 코코아분말, 당류가공품, 
step0.camera.png => 양파농축액, 과.채가공품, 땅콩버터(중국산), 대두레시틴, 정제소금, 새우, 밀크초콜릿, 육두구분말, 흑후추분말, 기타가공품, 5-리보뉴클레오티드이나트륨, 조개류(전복), 연꽃씨앗(심제외), L-글루탐산나트륨, 제삼


In [ ]:
# 셀 8. 3000개 사용 및 Train/Val 분할
random.seed(SEED)
random.shuffle(samples)

if len(samples) >= MAX_SAMPLES:
    samples = samples[:MAX_SAMPLES]
    print(f"사용 가능한 데이터가 {MAX_SAMPLES}개 이상이므로 셔플 후 {MAX_SAMPLES}개만 사용합니다.")
else:
    print(f"[경고] 사용 가능한 데이터가 {MAX_SAMPLES}개 미만입니다. 현재 데이터 전체를 사용합니다.")

split_index = int(len(samples) * TRAIN_RATIO)
if split_index <= 0 or split_index >= len(samples):
    raise RuntimeError("데이터가 너무 적어 train/val 분할이 불가능합니다.")

train_samples = samples[:split_index]
val_samples = samples[split_index:]

print("Total used:", len(samples))
print("Train count:", len(train_samples))
print("Val count:", len(val_samples))
print("Train ratio:", len(train_samples) / len(samples))
print("Val ratio:", len(val_samples) / len(samples))


사용 가능한 데이터가 3000개 이상이므로 셔플 후 3000개만 사용합니다.
Total used: 3000
Train count: 2400
Val count: 600
Train ratio: 0.8
Val ratio: 0.2


In [ ]:
# 셀 9. 이미지 전처리 함수와 Dataset 정의
def resize_long_side(image, long_side=1024):
    width, height = image.size
    current_long_side = max(width, height)
    if current_long_side <= long_side:
        return image
    scale = long_side / current_long_side
    new_size = (int(width * scale), int(height * scale))
    return image.resize(new_size, Image.Resampling.LANCZOS)

def preprocess_image(image, train=False):
    image = ImageOps.exif_transpose(image).convert("RGB")

    if USE_IMAGE_PREPROCESSING:
        # 1) Resize: 너무 큰 이미지를 줄여 학습/추론 안정화
        image = resize_long_side(image, RESIZE_LONG_SIDE)

        # 2) Contrast: 흐린 글자 대비 강화
        image = ImageEnhance.Contrast(image).enhance(CONTRAST_FACTOR)

        # 3) Denoise: 작은 노이즈 완화
        image = image.filter(ImageFilter.MedianFilter(size=DENOISE_SIZE))

        # 4) Sharpen: 글자 경계 조금 강화
        image = ImageEnhance.Sharpness(image).enhance(SHARPNESS_FACTOR)

    if train and USE_TRAIN_AUGMENTATION:
        # 선택적 약한 증강. 과하면 OCR이 망가질 수 있어서 매우 약하게만 적용.
        if random.random() < 0.3:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.9, 1.1))
        if random.random() < 0.3:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.9, 1.15))
        if random.random() < 0.2:
            image = image.rotate(random.uniform(-1.5, 1.5), expand=True, fillcolor=(255, 255, 255))

    return image

class FoodIngredientDataset(Dataset):
    def __init__(self, samples, train=False):
        self.samples = samples
        self.train = train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample["image_path"]).convert("RGB")
        image = preprocess_image(image, train=self.train)
        return {
            "image": image,
            "target_text": sample["target_text"],
            "filename": sample["filename"],
        }

train_dataset = FoodIngredientDataset(train_samples, train=True)
val_dataset = FoodIngredientDataset(val_samples, train=False)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))
print("USE_IMAGE_PREPROCESSING:", USE_IMAGE_PREPROCESSING)
print("USE_TRAIN_AUGMENTATION:", USE_TRAIN_AUGMENTATION)


Train dataset: 2400
Val dataset: 600
USE_IMAGE_PREPROCESSING: True
USE_TRAIN_AUGMENTATION: False


In [ ]:
# 셀 10. Processor와 Model 로드
processor = Pix2StructProcessor.from_pretrained(MODEL_NAME)
processor.image_processor.is_vqa = False

model = Pix2StructForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)

print("Model loaded:", MODEL_NAME)
print("processor.image_processor.is_vqa:", processor.image_processor.is_vqa)
print("Device:", next(model.parameters()).device)



Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

Model loaded: google/pix2struct-base
processor.image_processor.is_vqa: False
Device: cuda:0


In [ ]:
# 셀 11. Collate 함수 정의
def collate_fn(batch):
    images = [item["image"] for item in batch]
    target_texts = [item["target_text"] for item in batch]
    prompts = [PROMPT for _ in batch]

    encoding = processor(
        images=images,
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_patches=MAX_PATCHES,
    )

    labels = processor.tokenizer(
        target_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
    ).input_ids

    labels[labels == processor.tokenizer.pad_token_id] = -100
    encoding["labels"] = labels
    encoding["target_texts"] = target_texts
    encoding["filenames"] = [item["filename"] for item in batch]
    return encoding

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

print("Train loader batches:", len(train_loader))
print("Val loader batches:", len(val_loader))


Train loader batches: 2400
Val loader batches: 600


In [ ]:
# 셀 12. Smoke Batch 확인
smoke_batch = next(iter(train_loader))

print("Smoke batch keys:", smoke_batch.keys())
print("flattened_patches shape:", smoke_batch["flattened_patches"].shape)
print("attention_mask shape:", smoke_batch["attention_mask"].shape)
print("labels shape:", smoke_batch["labels"].shape)
print("target example:", smoke_batch["target_texts"][0])

model.train()
smoke_inputs = {
    "flattened_patches": smoke_batch["flattened_patches"].to(device),
    "attention_mask": smoke_batch["attention_mask"].to(device),
    "labels": smoke_batch["labels"].to(device),
}

with torch.cuda.amp.autocast(dtype=torch.float16):
    smoke_outputs = model(**smoke_inputs)
    smoke_loss = smoke_outputs.loss

print("Smoke loss:", float(smoke_loss.detach().cpu()))


Smoke batch keys: KeysView({'flattened_patches': tensor([[[ 1.0000,  1.0000, -1.5529,  ...,  0.1634,  0.2211, -0.0111],
         [ 1.0000,  2.0000, -0.9936,  ..., -0.0448, -0.0629, -0.7696],
         [ 1.0000,  3.0000,  1.1970,  ...,  0.1668, -0.4454, -0.7675],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]]), 'attention_mask': tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
    

/tmp/ipykernel_19704/1749954382.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Smoke loss: 14.390058517456055


In [ ]:
# 셀 13. Optimizer와 Scheduler 설정
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

num_update_steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
total_training_steps = EPOCHS * num_update_steps_per_epoch
warmup_steps = max(1, int(total_training_steps * 0.1))

scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

scaler = torch.cuda.amp.GradScaler()

print("EPOCHS:", EPOCHS)
print("EARLY_STOPPING_PATIENCE:", EARLY_STOPPING_PATIENCE)
print("BATCH_SIZE:", BATCH_SIZE)
print("ACCUMULATION_STEPS:", ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)


EPOCHS: 30
EARLY_STOPPING_PATIENCE: 4
BATCH_SIZE: 1
ACCUMULATION_STEPS: 4
Learning rate: 3e-05
Total training steps: 18000
Warmup steps: 1800


/tmp/ipykernel_19704/864279037.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
# 셀 14.
import json

CHECKPOINT_INFO_PATH = OUTPUT_DIR / "resume_info.json"
history_csv_path = OUTPUT_DIR / "train_history.csv"

start_epoch = 0
best_val_loss = float("inf")
patience_counter = 0
train_history = []

if BEST_MODEL_DIR.exists() and (BEST_MODEL_DIR / "config.json").exists() and CHECKPOINT_INFO_PATH.exists():
    print("=" * 60)
    print("🔄 구글 드라이브에서 기존 학습 가중치를 발견했습니다! 이어서 학습을 시작합니다.")

    # 1. 중간 저장된 가중치로 모델 교체 복구
    model = Pix2StructForConditionalGeneration.from_pretrained(BEST_MODEL_DIR)
    model.to(device)

    # 2. 이어할 에폭, 최고 loss, patience 정보 복구
    with open(CHECKPOINT_INFO_PATH, "r", encoding="utf-8") as f:
        checkpoint_info = json.load(f)
    start_epoch = checkpoint_info["epoch"]  # 저장 당시 완료된 에폭
    best_val_loss = checkpoint_info["best_val_loss"]
    patience_counter = checkpoint_info["patience_counter"]

    # 3. 기존 Loss 기록(History) 복구
    if history_csv_path.exists():
        train_history = pd.read_csv(history_csv_path).to_dict(orient="records")

    # 4. 스케줄러 강제 매칭 (지나간 스텝만큼 스케줄러를 당겨줍니다)
    # 1에폭당 업데이트 스텝 수 계산
    num_update_steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
    passed_steps = start_epoch * num_update_steps_per_epoch
    for _ in range(passed_steps):
        scheduler.step()

    print(f"▶️ [복구 완료] Epoch {start_epoch + 1}부터 재시작합니다. (기존 최고 Val Loss: {best_val_loss:.6f})")
    print("=" * 60)
else:
    print("🆕 기존 학습 기록이 없으므로 처음(Epoch 1)부터 새롭게 학습을 시작합니다.")

# -------------------------------------------------------------------

model.train()
global_step = start_epoch * math.ceil(len(train_loader) / ACCUMULATION_STEPS) if 'start_epoch' in locals() else 0

for epoch in range(start_epoch, EPOCHS):
    print("=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress_bar = tqdm(train_loader, desc=f"Training epoch {epoch + 1}")

    for step, batch in enumerate(progress_bar):
        inputs = {
            "flattened_patches": batch["flattened_patches"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
            "labels": batch["labels"].to(device),
        }

        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model(**inputs)
            loss = outputs.loss
            loss_for_backward = loss / ACCUMULATION_STEPS

        scaler.scale(loss_for_backward).backward()
        running_loss += float(loss.detach().cpu())

        should_update = (step + 1) % ACCUMULATION_STEPS == 0
        is_last_step = (step + 1) == len(train_loader)
        if should_update or is_last_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        avg_loss = running_loss / (step + 1)
        progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": scheduler.get_last_lr()[0]})

        if step == 0:
            print("First batch loss:", float(loss.detach().cpu()))

    epoch_avg_train_loss = running_loss / len(train_loader)
    epoch_avg_val_loss = compute_validation_loss(model, val_loader)

    history_row = {
        "epoch": epoch + 1,
        "train_loss": epoch_avg_train_loss,
        "val_loss": epoch_avg_val_loss,
        "lr": scheduler.get_last_lr()[0],
    }
    train_history.append(history_row)

    print(f"Epoch {epoch + 1} train loss: {epoch_avg_train_loss:.6f}")
    print(f"Epoch {epoch + 1} val loss  : {epoch_avg_val_loss:.6f}")

    # Early Stopping 및 저장 판단
    improved = epoch_avg_val_loss < (best_val_loss - MIN_DELTA)

    if improved:
        best_val_loss = epoch_avg_val_loss
        patience_counter = 0

        # 가중치 구글 드라이브에 실시간 저장
        BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(BEST_MODEL_DIR)
        processor.save_pretrained(BEST_MODEL_DIR)
        print(f"✅ Best model updated. best_val_loss={best_val_loss:.6f}")
    else:
        patience_counter += 1
        print(f"⚠️ Validation loss 개선 없음. patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    # [핵심] 개선 여부와 상관없이, 이번 에폭이 끝난 시점의 메타 정보를 항상 구글 드라이브에 백업 (튕겼을 때 이어하기용)
    checkpoint_meta = {
        "epoch": epoch + 1,
        "best_val_loss": best_val_loss,
        "patience_counter": patience_counter
    }
    with open(CHECKPOINT_INFO_PATH, "w", encoding="utf-8") as f:
        json.dump(checkpoint_meta, f, indent=4)

    # 실시간 History CSV 저장
    pd.DataFrame(train_history).to_csv(history_csv_path, index=False)

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print("🛑 EarlyStopping 발동: 과적합 방지를 위해 학습을 중단합니다.")
        break

# 최종 완료 저장
print("\n" + "="*60)
print("💾 최종 모델 및 학습 로그 저장 중...")
print("="*60)

FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"✅ Final model saved to: {FINAL_MODEL_DIR}")

history_df = pd.DataFrame(train_history)
print(f"✅ Train history saved to: {history_csv_path}")
print("📊 학습 로그 요약:")
print(history_df.to_string(index=False))

🆕 기존 학습 기록이 없으므로 처음(Epoch 1)부터 새롭게 학습을 시작합니다.
Epoch 1/30


Training epoch 1:   0%|          | 0/2400 [00:00<?, ?it/s]

/tmp/ipykernel_19704/2646934149.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


First batch loss: 13.796119689941406


Validation loss:   0%|          | 0/600 [00:00<?, ?it/s]

...

First batch loss: 0.7595609426498413


Validation loss:   0%|          | 0/600 [00:00<?, ?it/s]

Epoch 30 train loss: 0.854496
Epoch 30 val loss  : 0.822459
⚠️ Validation loss 개선 없음. patience 2/4

💾 최종 모델 및 학습 로그 저장 중...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final model saved to: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_final
✅ Train history saved to: /content/drive/MyDrive/pix2struct_food_v2_result/train_history.csv
📊 학습 로그 요약:
 epoch  train_loss  val_loss           lr
     1    6.755772  3.351530 1.053333e-05
     2    2.612638  1.116645 2.053333e-05
     3    1.285673  0.901389 2.999971e-05
     4    1.056490  0.857782 2.988748e-05
     5    0.986989  0.851490 2.957393e-05
     6    0.950093  0.839106 2.906328e-05
     7    0.927837  0.837876 2.836246e-05
     8    0.914739  0.835038 2.748093e-05
     9    0.905628  0.830552 2.643061e-05
    10    0.898775  0.831050 2.522572e-05
    11    0.891066  0.830981 2.388254e-05
    12    0.886635  0.825375 2.241924e-05
    13    0.882384  0.827294 2.085561e-05
    14    0.878137  0.826125 1.921279e-05
    15    0.874133  0.824509 1.751300e-05
    16    0.871607  0.825293 1.577923e-05
    17    0.869207  0.824216 1.403492e-05
    18    0.867470  0.825323 1.230366e-05
    

In [22]:
# 셀 15. 평가 함수 정의 + 출력 후처리
def normalize_text(text):
    text = text or ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" ,", ",")
    text = text.replace(", ", ", ")
    return text.strip()

def clean_generated_text(text):
    """선배 피드백 반영: <s>, </s>, <pad> 같은 특수 토큰 제거"""
    text = text or ""
    special_tokens = [
        "<s>", "</s>", "<pad>", "<unk>",
        processor.tokenizer.pad_token if processor.tokenizer.pad_token else "",
        processor.tokenizer.eos_token if processor.tokenizer.eos_token else "",
        processor.tokenizer.bos_token if processor.tokenizer.bos_token else "",
    ]
    for token in special_tokens:
        if token:
            text = text.replace(token, "")
    text = clean_ingredient_text(text)
    text = normalize_text(text)
    return text

def levenshtein_distance(a, b):
    a = a or ""
    b = b or ""
    previous_row = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current_row = [i]
        for j, char_b in enumerate(b, start=1):
            insert_cost = current_row[j - 1] + 1
            delete_cost = previous_row[j] + 1
            replace_cost = previous_row[j - 1] + (0 if char_a == char_b else 1)
            current_row.append(min(insert_cost, delete_cost, replace_cost))
        previous_row = current_row
    return previous_row[-1]

def calculate_sample_cer(prediction, reference):
    reference = reference or ""
    if len(reference) == 0:
        return 0.0
    return levenshtein_distance(prediction, reference) / len(reference)

def calculate_cer(predictions, references):
    total_distance = 0
    total_chars = 0
    for pred, ref in zip(predictions, references):
        total_distance += levenshtein_distance(pred, ref)
        total_chars += len(ref)
    return total_distance / total_chars if total_chars > 0 else 0.0

def extract_allergy_keywords(text):
    return {keyword for keyword in ALLERGY_KEYWORDS if keyword in text}

def calculate_allergy_metrics(predictions, references):
    true_positive = 0
    false_positive = 0
    false_negative = 0
    for pred, ref in zip(predictions, references):
        pred_set = extract_allergy_keywords(pred)
        ref_set = extract_allergy_keywords(ref)
        true_positive += len(pred_set & ref_set)
        false_positive += len(pred_set - ref_set)
        false_negative += len(ref_set - pred_set)

    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0.0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "allergy_precision": precision,
        "allergy_recall": recall,
        "allergy_f1": f1,
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
    }

def classify_error_type(prediction, reference):
    pred_allergy = extract_allergy_keywords(prediction)
    ref_allergy = extract_allergy_keywords(reference)

    if not prediction.strip():
        return "empty_prediction"
    if any("\u4e00" <= ch <= "\u9fff" for ch in prediction):
        return "chinese_or_hanja_token"
    if len(prediction) < max(5, len(reference) * 0.4):
        return "too_short_or_missing_text"
    if len(prediction) > len(reference) * 1.8:
        return "too_long_or_hallucination"
    if ref_allergy - pred_allergy:
        return "allergy_keyword_missing"
    if pred_allergy - ref_allergy:
        return "allergy_keyword_false_positive"
    return "general_ocr_error"

print("Evaluation functions ready.")


Evaluation functions ready.


In [23]:
# 셀 16. Best Model 로드 후 Generate 평가 실행
if BEST_MODEL_DIR.exists():
    print("Best model을 로드해서 평가합니다:", BEST_MODEL_DIR)
    model = Pix2StructForConditionalGeneration.from_pretrained(BEST_MODEL_DIR).to(device)
else:
    print("[경고] Best model dir이 없어 현재 메모리의 model로 평가합니다.")

model.eval()
all_eval_results = {}

for num_beams in EVAL_NUM_BEAMS_LIST:
    print("=" * 60)
    print(f"Evaluating with num_beams = {num_beams}")
    print("=" * 60)

    predictions = []
    references = []
    filenames = []
    sample_cers = []
    error_types = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Evaluating beam={num_beams}"):
            generation_inputs = {
                "flattened_patches": batch["flattened_patches"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }
            generated_ids = model.generate(
                **generation_inputs,
                max_new_tokens=MAX_GENERATION_LENGTH,
                num_beams=num_beams,
            )

            decoded_predictions = processor.batch_decode(generated_ids, skip_special_tokens=False)

            for pred, ref, filename in zip(decoded_predictions, batch["target_texts"], batch["filenames"]):
                clean_pred = clean_generated_text(pred)
                clean_ref = normalize_text(ref)

                predictions.append(clean_pred)
                references.append(clean_ref)
                filenames.append(filename)
                sample_cers.append(calculate_sample_cer(clean_pred, clean_ref))
                error_types.append(classify_error_type(clean_pred, clean_ref))

    cer = calculate_cer(predictions, references)
    allergy_metrics = calculate_allergy_metrics(predictions, references)

    all_eval_results[num_beams] = {
        "num_beams": num_beams,
        "predictions": predictions,
        "references": references,
        "filenames": filenames,
        "sample_cers": sample_cers,
        "error_types": error_types,
        "cer": cer,
        "allergy_metrics": allergy_metrics,
    }

    print("=" * 60)
    print(f"Evaluation Results - beam={num_beams}")
    print("=" * 60)
    print("CER:", cer)
    print("Allergy Keyword Precision:", allergy_metrics["allergy_precision"])
    print("Allergy Keyword Recall:", allergy_metrics["allergy_recall"])
    print("Allergy Keyword F1:", allergy_metrics["allergy_f1"])
    print("TP:", allergy_metrics["true_positive"])
    print("FP:", allergy_metrics["false_positive"])
    print("FN:", allergy_metrics["false_negative"])

    error_df = pd.DataFrame({"error_type": error_types})
    print("\nError type count:")
    print(error_df["error_type"].value_counts())

    print("\n예측 샘플:")
    for i in range(min(5, len(predictions))):
        print("-" * 60)
        print("File:", filenames[i])
        print("CER :", sample_cers[i])
        print("ERR :", error_types[i])
        print("REF :", references[i])
        print("PRED:", predictions[i])

print("Beam 1, Beam 3 평가 완료")


Best model을 로드해서 평가합니다: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_best


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

Evaluating with num_beams = 1


Evaluating beam=1:   0%|          | 0/600 [00:00<?, ?it/s]

Evaluation Results - beam=1
CER: 0.7617249719780544
Allergy Keyword Precision: 0.5016666666666667
Allergy Keyword Recall: 0.162658740880843
Allergy Keyword F1: 0.245664150173434
TP: 602
FP: 598
FN: 3099

Error type count:
error_type
allergy_keyword_missing    599
general_ocr_error            1
Name: count, dtype: int64

예측 샘플:
------------------------------------------------------------
File: step0.camera.png
CER : 0.770949720670391
ERR : allergy_keyword_missing
REF : 카레분, 쇠고기, 쇼트닝, 땅콩버터(중국산), 복숭아, 닭고기추출물분말, 난황액(계란:국산), 흑후추가루, 분말결정포도당, 혼합제제, 이산화규소, 양파농축액, 쇠고기맛베이스, 복합조미식품, 백설탕, 복합조미식품, 조개류(굴), 정제수, 흑후추분말, 대두레시틴, 정제소금, 토코페롤(혼합형), 복합조미식품, 향료, 혼합제제
PRED: 고기출물분,, 지분유(우유:네란드산),,,,, 고기,,, 지대두,, 지고기, 지(지고기:국산),, 지고기, 지고기, 지고기, 지고기, 지고기, 지고기, 지고기, 지고기, 지대두, 지고기
------------------------------------------------------------
File: step0.camera.png
CER : 0.7445652173913043
ERR : allergy_keyword_missing
REF : D-소비톨액, 토코페롤(혼합형), 사과산, 초콜릿, 쇠고기맛베이스, 혼합분유, 땅콩, 강낭콩, 땅콩버터(중국산), 게추출물, 기타가공품, 정제소금, 새우분말, 볶음양

Evaluating beam=3:   0%|          | 0/600 [00:00<?, ?it/s]

Evaluation Results - beam=3
CER: 0.7839557154936778
Allergy Keyword Precision: 0.5133858267716536
Allergy Keyword Recall: 0.0880843015401243
Allergy Keyword F1: 0.15036900369003692
TP: 326
FP: 309
FN: 3375

Error type count:
error_type
allergy_keyword_missing    600
Name: count, dtype: int64

예측 샘플:
------------------------------------------------------------
File: step0.camera.png
CER : 0.7374301675977654
ERR : allergy_keyword_missing
REF : 카레분, 쇠고기, 쇼트닝, 땅콩버터(중국산), 복숭아, 닭고기추출물분말, 난황액(계란:국산), 흑후추가루, 분말결정포도당, 혼합제제, 이산화규소, 양파농축액, 쇠고기맛베이스, 복합조미식품, 백설탕, 복합조미식품, 조개류(굴), 정제수, 흑후추분말, 대두레시틴, 정제소금, 토코페롤(혼합형), 복합조미식품, 향료, 혼합제제
PRED: 고기출물분, 혼합제제, 복합조미식품, 지분유(우유:네란드산), 혼합제제, 혼합제제, 고기출물분, 복합조미식품, 혼합제제, 혼합제제, 고기출물분, 복합조미식품, 혼합제제, 고기출물분, 고기출물분, 복합조미식품, 혼합제제, 고기출물분, 혼합제제, 혼합제제, 고기출물분, 고기출물분, 고기출물분
------------------------------------------------------------
File: step0.camera.png
CER : 0.7391304347826086
ERR : allergy_keyword_missing
REF : D-소비톨액, 토코페롤(혼합형), 사과산, 초콜릿, 쇠고기맛베이스, 혼합분유, 땅콩, 강낭콩, 땅콩버터(중국산

In [26]:
# ============================================================

import pandas as pd
import torch
from tqdm import tqdm


if BEST_MODEL_DIR.exists():
    print("수정한 파라미터로 Best model을 다시 로드", BEST_MODEL_DIR)
    model = Pix2StructForConditionalGeneration.from_pretrained(BEST_MODEL_DIR).to(device)
else:
    print("경로가 없")

model.eval()
all_eval_results = {}

FAST_EVAL_BEAMS = [1]

for num_beams in FAST_EVAL_BEAMS:
    print("=" * 60)
    print(f"Evaluating with num_beams = {num_beams}")
    print("=" * 60)

    predictions = []
    references = []
    filenames = []
    sample_cers = []
    error_types = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Evaluating beam={num_beams}"):
            generation_inputs = {
                "flattened_patches": batch["flattened_patches"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }


            generated_ids = model.generate(
                **generation_inputs,
                max_new_tokens=MAX_GENERATION_LENGTH,
                num_beams=num_beams,
                repetition_penalty=2.5,          # 똑같은 단어 반복 시 패널티 부여 (뇌절 차단)
                no_repeat_ngram_size=3,          # 3글자 이상 연속 중복 절대 금지
                early_stopping=True              # 문장이 끝나면 즉시 연산 종료 (속도 획기적 단축)
            )

            decoded_predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)

            for pred, ref, filename in zip(decoded_predictions, batch["target_texts"], batch["filenames"]):
                # 깨끗해진 텍스트를 공백만 제거하여 바인딩
                clean_pred = pred.strip()
                clean_ref = normalize_text(ref)  # 기존 공통 함수 유지

                predictions.append(clean_pred)
                references.append(clean_ref)
                filenames.append(filename)

                # 지표 계산 및 에러 타입 분류 (기존 함수 활용)
                sample_cers.append(calculate_sample_cer(clean_pred, clean_ref))
                error_types.append(classify_error_type(clean_pred, clean_ref))

    # 최종 스코어 산출
    cer = calculate_cer(predictions, references)
    allergy_metrics = calculate_allergy_metrics(predictions, references)

    all_eval_results[num_beams] = {
        "num_beams": num_beams,
        "predictions": predictions,
        "references": references,
        "filenames": filenames,
        "sample_cers": sample_cers,
        "error_types": error_types,
        "cer": cer,
        "allergy_metrics": allergy_metrics,
    }

    print("=" * 60)
    print(f"📊 [개선 완료] Evaluation Results - beam={num_beams}")
    print("=" * 60)
    print("CER (글자 에러율):", cer)
    print("Allergy Keyword Precision (정밀도):", allergy_metrics["allergy_precision"])
    print("Allergy Keyword Recall (재현율):", allergy_metrics["allergy_recall"])
    print("Allergy Keyword F1 (종합 점수):", allergy_metrics["allergy_f1"])
    print(f"정답 매칭 수 (TP): {allergy_metrics['true_positive']} | 오답 수 (FP): {allergy_metrics['false_positive']}")

    error_df = pd.DataFrame({"error_type": error_types})
    print("\n에러 유형 분포:")
    print(error_df["error_type"].value_counts())

    print("\n👀 실제 예측결과 샘플 확인:")
    for i in range(min(5, len(predictions))):
        print("-" * 60)
        print("File:", filenames[i])
        print("CER :", sample_cers[i])
        print("ERR :", error_types[i])
        print("REF (진짜정답):", references[i])
        print("PRED(개선된AI):", predictions[i])

print("\n완료되었습니다!")

수정한 파라미터로 Best model을 다시 로드 /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_best


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

Evaluating with num_beams = 1


Evaluating beam=1: 100%|██████████| 600/600 [18:39<00:00,  1.87s/it]


📊 [개선 완료] Evaluation Results - beam=1
CER (글자 에러율): 0.8433032466127859
Allergy Keyword Precision (정밀도): 0.42038216560509556
Allergy Keyword Recall (재현율): 0.08916509051607674
Allergy Keyword F1 (종합 점수): 0.14712438698172092
정답 매칭 수 (TP): 330 | 오답 수 (FP): 455

에러 유형 분포:
error_type
allergy_keyword_missing      494
too_short_or_missing_text    106
Name: count, dtype: int64

👀 실제 예측결과 샘플 확인:
------------------------------------------------------------
File: step0.camera.png
CER : 0.7988826815642458
ERR : allergy_keyword_missing
REF (진짜정답): 카레분, 쇠고기, 쇼트닝, 땅콩버터(중국산), 복숭아, 닭고기추출물분말, 난황액(계란:국산), 흑후추가루, 분말결정포도당, 혼합제제, 이산화규소, 양파농축액, 쇠고기맛베이스, 복합조미식품, 백설탕, 복합조미식품, 조개류(굴), 정제수, 흑후추분말, 대두레시틴, 정제소금, 토코페롤(혼합형), 복합조미식품, 향료, 혼합제제
PRED(개선된AI): <s>고기, 정제소금(국산), 아세설합, 조개류혼물분유, 소두구분 kabanay, 고수분조절대, L-타민루과당, 우연나트 Спољашње, 마가린, 유청분 munisipyo, 연근, 기위치즈, 물<0x44>, 게
------------------------------------------------------------
File: step0.camera.png
CER : 0.7880434782608695
ERR : allergy_keyword_missing
REF (진짜정답

In [28]:
#블랙이미지를 주고 테스트. 같은 값들이 계속 나오는것에 대한 의문
import torch

model.eval()
with torch.no_grad():
    # 1. val_loader에서 딱 첫 번째 배치 하나만 가져옵니다.
    for batch in val_loader:
        # 2. 이미지 입력(flattened_patches)을 전부 0(완전 까만 화면)으로 강제 초기화.
        black_patches = torch.zeros_like(batch["flattened_patches"]).to(device)
        attention_mask = batch["attention_mask"].to(device)

        # 3. 까만 화면을 주고 예측
        generated_ids = model.generate(
            flattened_patches=black_patches,
            attention_mask=attention_mask,
            max_new_tokens=MAX_GENERATION_LENGTH,
            num_beams=1,
            repetition_penalty=2.5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

        decoded_predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)

        print("=" * 60)
        print("완전 까만 화면을 줬을 때 AI의 반응")
        print("=" * 60)
        for i, pred in enumerate(decoded_predictions[:3]):
            print(f"Sample {i+1} PRED: {pred.strip()}")
        break # 첫 배치만 보고 종료

완전 까만 화면을 줬을 때 AI의 반응
Sample 1 PRED: 조 물 건 고 변 정, 스 대 사 우 제 마 알 아 천 가 게 소 전 분 설 당 올가 오 다시 연 매 유 기 D 이 구분 - L 양 해 강과 무조고 초수 말제소스 생 DL 바로치유(전복합식러코트은 비타이산출대다모레지버-개류향후미메밀),육두구정토마나카오세설시리즈:5'란드크어혼자간심도우근적배포.계료위물성이나결연인파난호사단비스트 조청공기 조강신투매일에 물외 물 물 물 조금씨 물 물 스테레이 조아<s> 조국양 물 물 함 물 물 건 물 물 정화 물 물, 진네 물 물 사 조 물 물 C 물 물 고린 물 물 대 물 물 제 물 물 우 물 물 보 물 물


In [29]:
# 💡 [새 셸] 허깅페이스 순정 원본 모델 테스트 코드
from transformers import Pix2StructForConditionalGeneration

print("🔄 학습 전 순정 원본 Pix2Struct 모델을 로드합니다...")
raw_model = Pix2StructForConditionalGeneration.from_pretrained("google/pix2struct-base").to(device)
raw_model.eval()

with torch.no_grad():
    for batch in val_loader:
        generation_inputs = {
            "flattened_patches": batch["flattened_patches"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }

        generated_ids = raw_model.generate(
            **generation_inputs,
            max_new_tokens=MAX_GENERATION_LENGTH,
            num_beams=1,
            repetition_penalty=2.5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

        decoded_predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)

        print("=" * 60)
        print("🍏 [순정 모델 결과] 학습 전 원본 모델의 반응")
        print("=" * 60)
        for i, (pred, ref) in enumerate(zip(decoded_predictions[:3], batch["target_texts"][:3])):
            print(f"Sample {i+1} REF : {ref.strip()[:30]}...")
            print(f"Sample {i+1} PRED: {pred.strip()}")
        break

🔄 학습 전 순정 원본 Pix2Struct 모델을 로드합니다...


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

🍏 [순정 모델 결과] 학습 전 원본 모델의 반응
Sample 1 REF : 카레분, 쇠고기, 쇼트닝, 땅콩버터(중국산), 복숭아,...
Sample 1 PRED: <img_src=cropped-IMG_0219>


In [30]:
# 셀 17. 최종 모델과 결과 저장
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

for num_beams, eval_result in all_eval_results.items():
    predictions = eval_result["predictions"]
    references = eval_result["references"]
    filenames = eval_result["filenames"]
    sample_cers = eval_result["sample_cers"]
    error_types = eval_result["error_types"]
    cer = eval_result["cer"]
    allergy_metrics = eval_result["allergy_metrics"]

    result_json_path = OUTPUT_DIR / f"pix2struct_v2_eval_results_beam{num_beams}.json"
    prediction_jsonl_path = OUTPUT_DIR / f"pix2struct_v2_predictions_beam{num_beams}.jsonl"
    prediction_csv_path = OUTPUT_DIR / f"pix2struct_v2_predictions_beam{num_beams}.csv"
    error_summary_csv_path = OUTPUT_DIR / f"pix2struct_v2_error_summary_beam{num_beams}.csv"

    result_summary = {
        "model_name": MODEL_NAME,
        "prompt": PROMPT,
        "version": "pix2struct_v2",
        "num_beams": num_beams,
        "max_samples": MAX_SAMPLES,
        "epochs_requested": EPOCHS,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "gradient_clip_norm": GRAD_CLIP_NORM,
        "use_image_preprocessing": USE_IMAGE_PREPROCESSING,
        "use_train_augmentation": USE_TRAIN_AUGMENTATION,
        "resize_long_side": RESIZE_LONG_SIDE,
        "contrast_factor": CONTRAST_FACTOR,
        "sharpness_factor": SHARPNESS_FACTOR,
        "denoise_size": DENOISE_SIZE,
        "train_count": len(train_samples),
        "val_count": len(val_samples),
        "cer": cer,
        "allergy_keyword_precision": allergy_metrics["allergy_precision"],
        "allergy_keyword_recall": allergy_metrics["allergy_recall"],
        "allergy_keyword_f1": allergy_metrics["allergy_f1"],
        "true_positive": allergy_metrics["true_positive"],
        "false_positive": allergy_metrics["false_positive"],
        "false_negative": allergy_metrics["false_negative"],
        "allergy_keywords": ALLERGY_KEYWORDS,
        "train_history": train_history,
    }

    with open(result_json_path, "w", encoding="utf-8") as f:
        json.dump(result_summary, f, ensure_ascii=False, indent=2)

    rows = []
    with open(prediction_jsonl_path, "w", encoding="utf-8") as f:
        for filename, ref, pred, sample_cer, error_type in zip(
            filenames, references, predictions, sample_cers, error_types
        ):
            row = {
                "file_name": filename,
                "target": ref,
                "prediction": pred,
                "cer": sample_cer,
                "error_type": error_type,
                "reference_allergy_keywords": sorted(list(extract_allergy_keywords(ref))),
                "prediction_allergy_keywords": sorted(list(extract_allergy_keywords(pred))),
            }
            rows.append(row)
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(prediction_csv_path, index=False, encoding="utf-8-sig")

    error_summary_df = (
        pred_df.groupby("error_type")
        .agg(count=("file_name", "count"), mean_cer=("cer", "mean"))
        .reset_index()
        .sort_values("count", ascending=False)
    )
    error_summary_df.to_csv(error_summary_csv_path, index=False, encoding="utf-8-sig")

    print("저장 완료:", result_json_path)
    print("예측 JSONL 저장:", prediction_jsonl_path)
    print("예측 CSV 저장:", prediction_csv_path)
    print("오류 요약 CSV 저장:", error_summary_csv_path)

zip_file = shutil.make_archive("/content/pix2struct_food_v2_result", "zip", OUTPUT_DIR)
print("Best model:", BEST_MODEL_DIR)
print("Final model:", FINAL_MODEL_DIR)
print("Zipped output:", zip_file)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_eval_results_beam1.json
예측 JSONL 저장: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_predictions_beam1.jsonl
예측 CSV 저장: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_predictions_beam1.csv
오류 요약 CSV 저장: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_error_summary_beam1.csv
Best model: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_best
Final model: /content/drive/MyDrive/pix2struct_food_v2_result/pix2struct_v2_final
Zipped output: /content/pix2struct_food_v2_result.zip
